%md
# 05 · Gold account features
Builds one row per account with network and behaviour features for alerting. Reads silver transactions WITHOUT the is_laundering label and excludes self-transfers from network features.

In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.aml_silver"
GOLD = "workspace.aml_gold"

txns = spark.table(f"{SILVER}.transactions").select(
    "txn_id",
    "txn_timestamp",
    "from_account_id",
    "to_account_id",
    "amount_paid",
    "payment_currency",
    "amount_received",
    "receiving_currency",
    "payment_format",
    "is_self_transfer",
)

network_txns = txns.filter(~F.col("is_self_transfer"))

print(f"All transactions:     {txns.count():,}")
print(f"Network transactions: {network_txns.count():,}")

In [0]:
display(
    txns.groupBy("payment_currency")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
cross_usd = (
    txns
    .filter(
        (F.col("payment_currency") == "US Dollar")
        & (F.col("receiving_currency") != "US Dollar")
        & (F.col("amount_paid") > 0)
    )
    .withColumn("rate", F.col("amount_received") / F.col("amount_paid"))
)

display(
    cross_usd.groupBy("receiving_currency")
    .agg(
        F.count("*").alias("txns"),
        F.percentile_approx("rate", 0.5).alias("median_rate"),
        F.min("rate").alias("min_rate"),
        F.max("rate").alias("max_rate"),
    )
    .orderBy("receiving_currency")
)

In [0]:
fx = (
    cross_usd
    .groupBy(F.col("receiving_currency").alias("currency"))
    .agg(F.percentile_approx("rate", 0.5).cast("double").alias("units_per_usd"))
    .union(spark.createDataFrame([("US Dollar", 1.0)], ["currency", "units_per_usd"]))
)

display(fx.orderBy("currency"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

(
    fx.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD}.fx_rates")
)

In [0]:
fx_paid = fx.select(
    F.col("currency").alias("payment_currency"),
    F.col("units_per_usd").alias("paid_rate"),
)
fx_received = fx.select(
    F.col("currency").alias("receiving_currency"),
    F.col("units_per_usd").alias("received_rate"),
)

txns_usd = (
    txns
    .join(F.broadcast(fx_paid), on="payment_currency", how="left")
    .join(F.broadcast(fx_received), on="receiving_currency", how="left")
    .withColumn("amount_paid_usd", F.col("amount_paid").cast("double") / F.col("paid_rate"))
    .withColumn("amount_received_usd", F.col("amount_received").cast("double") / F.col("received_rate"))
    .drop("paid_rate", "received_rate")
)

In [0]:
missing_usd = txns_usd.filter(
    F.col("amount_paid_usd").isNull() | F.col("amount_received_usd").isNull()
).count()

print(f"Rows: {txns_usd.count():,}")
print(f"Rows missing a USD amount: {missing_usd:,}")

cross_check = (
    txns_usd
    .filter(
        (F.col("payment_currency") != F.col("receiving_currency"))
        & (F.col("amount_received_usd") > 1)
    )
    .withColumn("ratio", F.col("amount_paid_usd") / F.col("amount_received_usd"))
)

display(
    cross_check.agg(
        F.count("*").alias("cross_currency_txns"),
        F.percentile_approx("ratio", 0.5).alias("median_ratio"),
        F.avg((F.abs(F.col("ratio") - 1) < 0.02).cast("int")).alias("share_within_2pct"),
    )
)

In [0]:
network_usd = txns_usd.filter(~F.col("is_self_transfer"))

outgoing = (
    network_usd
    .groupBy(F.col("from_account_id").alias("account_id"))
    .agg(
        F.count("*").alias("out_txn_count"),
        F.countDistinct("to_account_id").alias("out_degree"),
        F.sum("amount_paid_usd").alias("out_amount_usd"),
        F.max("amount_paid_usd").alias("out_max_usd"),
        F.countDistinct("payment_format").alias("out_payment_formats"),
        F.min("txn_timestamp").alias("first_out_at"),
        F.max("txn_timestamp").alias("last_out_at"),
    )
)

incoming = (
    network_usd
    .groupBy(F.col("to_account_id").alias("account_id"))
    .agg(
        F.count("*").alias("in_txn_count"),
        F.countDistinct("from_account_id").alias("in_degree"),
        F.sum("amount_received_usd").alias("in_amount_usd"),
        F.max("amount_received_usd").alias("in_max_usd"),
        F.min("txn_timestamp").alias("first_in_at"),
        F.max("txn_timestamp").alias("last_in_at"),
    )
)

In [0]:
accounts = spark.table(f"{SILVER}.accounts").select("account_id", "bank_id", "entity_id")

zero_cols = [
    "out_txn_count", "out_degree", "out_amount_usd", "out_max_usd", "out_payment_formats",
    "in_txn_count", "in_degree", "in_amount_usd", "in_max_usd",
]

features = (
    accounts
    .join(outgoing, on="account_id", how="left")
    .join(incoming, on="account_id", how="left")
    .fillna(0, subset=zero_cols)
)

In [0]:
totals = features.agg(
    F.count("*").alias("accounts"),
    F.sum("out_txn_count").alias("total_out"),
    F.sum("in_txn_count").alias("total_in"),
).first()

print(f"Accounts:  {totals['accounts']:,}")
print(f"Total out: {totals['total_out']:,}")
print(f"Total in:  {totals['total_in']:,}")

assert totals["accounts"] == 518581, "Account count changed"
assert totals["total_out"] == 4487133, "Outgoing counts do not add up to network transactions"
assert totals["total_in"] == 4487133, "Incoming counts do not add up to network transactions"

print("Quality gate passed")

display(
    features.select(
        F.percentile_approx("out_degree", [0.5, 0.9, 0.99]).alias("out_degree_p50_p90_p99"),
        F.max("out_degree").alias("out_degree_max"),
        F.percentile_approx("in_degree", [0.5, 0.9, 0.99]).alias("in_degree_p50_p90_p99"),
        F.max("in_degree").alias("in_degree_max"),
    )
)

In [0]:
daily_out = (
    network_usd
    .groupBy(F.col("from_account_id").alias("account_id"), F.to_date("txn_timestamp").alias("txn_date"))
    .count()
    .groupBy("account_id")
    .agg(
        F.max("count").alias("out_max_txns_per_day"),
        F.count("*").alias("out_active_days"),
    )
)

daily_in = (
    network_usd
    .groupBy(F.col("to_account_id").alias("account_id"), F.to_date("txn_timestamp").alias("txn_date"))
    .count()
    .groupBy("account_id")
    .agg(
        F.max("count").alias("in_max_txns_per_day"),
        F.count("*").alias("in_active_days"),
    )
)

In [0]:
velocity_cols = ["out_max_txns_per_day", "out_active_days", "in_max_txns_per_day", "in_active_days"]

account_features = (
    features
    .join(daily_out, on="account_id", how="left")
    .join(daily_in, on="account_id", how="left")
    .fillna(0, subset=velocity_cols)
    .withColumn(
        "pass_through_ratio",
        F.when(
            (F.col("in_amount_usd") > 0) & (F.col("out_amount_usd") > 0),
            F.least("in_amount_usd", "out_amount_usd") / F.greatest("in_amount_usd", "out_amount_usd"),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn("net_flow_usd", F.col("in_amount_usd") - F.col("out_amount_usd"))
    .withColumn("first_seen_at", F.least("first_in_at", "first_out_at"))
    .withColumn("last_seen_at", F.greatest("last_in_at", "last_out_at"))
    .withColumn(
        "active_span_hours",
        (F.unix_timestamp("last_seen_at") - F.unix_timestamp("first_seen_at")) / 3600,
    )
)

In [0]:
print(f"Rows: {account_features.count():,}")

both_ways = account_features.filter((F.col("in_amount_usd") > 0) & (F.col("out_amount_usd") > 0))

display(
    both_ways.select(
        F.count("*").alias("accounts_in_and_out"),
        F.percentile_approx("pass_through_ratio", [0.5, 0.9, 0.99]).alias("pass_through_p50_p90_p99"),
        F.percentile_approx("out_max_txns_per_day", [0.5, 0.9, 0.99]).alias("out_per_day_p50_p90_p99"),
    )
)

display(
    account_features
    .orderBy(F.desc("out_degree"))
    .select(
        "account_id", "entity_id", "out_degree", "in_degree",
        "out_amount_usd", "in_amount_usd", "pass_through_ratio", "out_max_txns_per_day",
    )
    .limit(10)
)

In [0]:
HUB_LIMIT = 100

hubs = account_features.filter(
    (F.col("out_degree") > HUB_LIMIT) | (F.col("in_degree") > HUB_LIMIT)
).select("account_id")

edges = (
    network_usd
    .select(F.col("from_account_id").alias("src"), F.col("to_account_id").alias("dst"))
    .distinct()
    .join(hubs.withColumnRenamed("account_id", "src"), on="src", how="left_anti")
    .join(hubs.withColumnRenamed("account_id", "dst"), on="dst", how="left_anti")
)

print(f"Hub accounts excluded: {hubs.count():,}")
print(f"Links in the search:   {edges.count():,}")

In [0]:
e1 = edges.alias("e1")
e2 = edges.alias("e2")

cycle2_accounts = (
    e1.join(
        e2,
        (F.col("e1.dst") == F.col("e2.src")) & (F.col("e2.dst") == F.col("e1.src")),
    )
    .select(F.col("e1.src").alias("account_id"))
    .distinct()
)

print(f"Accounts in a 2-hop loop: {cycle2_accounts.count():,}")

In [0]:
e3 = edges.alias("e3")

cycle3_accounts = (
    e1.join(e2, F.col("e1.dst") == F.col("e2.src"))
    .filter(F.col("e2.dst") != F.col("e1.src"))
    .join(
        e3,
        (F.col("e2.dst") == F.col("e3.src")) & (F.col("e3.dst") == F.col("e1.src")),
    )
    .select(F.col("e1.src").alias("account_id"))
    .distinct()
)

print(f"Accounts in a 3-hop loop: {cycle3_accounts.count():,}")

In [0]:
gold_accounts = (
    account_features
    .join(cycle2_accounts.withColumn("in_cycle_2", F.lit(True)), on="account_id", how="left")
    .join(cycle3_accounts.withColumn("in_cycle_3", F.lit(True)), on="account_id", how="left")
    .join(hubs.withColumn("is_hub", F.lit(True)), on="account_id", how="left")
    .fillna(False, subset=["in_cycle_2", "in_cycle_3", "is_hub"])
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
rows = gold_accounts.count()
unique_ids = gold_accounts.select("account_id").distinct().count()
flags = gold_accounts.agg(
    F.sum(F.col("in_cycle_2").cast("int")).alias("cycle_2"),
    F.sum(F.col("in_cycle_3").cast("int")).alias("cycle_3"),
    F.sum(F.col("is_hub").cast("int")).alias("hubs"),
).first()

print(f"Rows:            {rows:,}")
print(f"Unique accounts: {unique_ids:,}")
print(f"In 2-hop loop:   {flags['cycle_2']:,}")
print(f"In 3-hop loop:   {flags['cycle_3']:,}")
print(f"Hubs:            {flags['hubs']:,}")

forbidden = [c for c in gold_accounts.columns if "laundering" in c.lower() or "pattern" in c.lower()]

assert rows == 518581, "Account count changed"
assert unique_ids == rows, "account_id is not unique"
assert flags["cycle_2"] == cycle2_accounts.count(), "2-hop flags lost in join"
assert flags["cycle_3"] == cycle3_accounts.count(), "3-hop flags lost in join"
assert flags["hubs"] == hubs.count(), "Hub flags lost in join"
assert not forbidden, f"Label columns found in Gold: {forbidden}"

print("Quality gate passed")

(
    gold_accounts.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD}.account_features")
)